<a href="https://colab.research.google.com/github/dzidz1/Freeuni_ML_Facial_Expression_Recognition/blob/main/01_data_and_baseline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Environment Setup

In [4]:
import torch
print("GPU available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

GPU available: True
Device: Tesla T4


In [5]:
!pip install -q --upgrade kaggle

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.8/132.8 kB 12.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.0/231.0 kB 26.3 MB/s eta 0:00:00


## Data Acquisition

In [6]:
from google.colab import userdata
import os
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

In [7]:
!kaggle competitions download -c challenges-in-representation-learning-facial-expression-recognition-challenge
!unzip -q -o challenges-in-representation-learning-facial-expression-recognition-challenge.zip
!ls -la

100% 285M/285M [00:17<00:00, 16.9MB/s]

total 974056
drwxr-xr-x 1 root root      4096 Jun 14 18:39 .
drwxr-xr-x 1 root root      4096 Jun 14 17:58 ..
-rw-r--r-- 1 root root 299063632 Dec 11  2019 challenges-in-representation-learning-facial-expression-recognition-challenge.zip
drwxr-xr-x 4 root root      4096 Jun  4 13:39 .config
-rw-r--r-- 1 root root      7178 Dec 11  2019 example_submission.csv
-rw-r--r-- 1 root root  96433867 Dec 11  2019 fer2013.tar.gz
-rw-r--r-- 1 root root 301072768 Dec 11  2019 icml_face_data.csv
drwxr-xr-x 1 root root      4096 Jun  4 13:39 sample_data
-rw-r--r-- 1 root root  60125203 Dec 11  2019 test.csv
-rw-r--r-- 1 root root 240699943 Dec 11  2019 train.csv


## Exploratory Data Analysis

In [8]:
import pandas as pd

df = pd.read_csv('icml_face_data.csv')
df.columns = df.columns.str.strip()

print("Shape:", df.shape)
print("Columns:", list(df.columns))
print()
print(df.head())
print()
print("Official split counts:")
print(df['Usage'].value_counts())
print()
print("Emotion counts (0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral):")
print(df['emotion'].value_counts().sort_index())

Shape: (35887, 3)
Columns: ['emotion', 'Usage', 'pixels']

   emotion     Usage                                             pixels
0        0  Training  70 80 82 72 58 58 60 63 54 58 60 48 89 115 121...
1        0  Training  151 150 147 155 148 133 111 140 170 174 182 15...
2        2  Training  231 212 156 164 174 138 161 173 182 200 106 38...
3        4  Training  24 32 36 30 32 23 19 20 30 41 21 22 32 34 21 1...
4        6  Training  4 0 0 0 0 0 0 0 0 0 0 0 3 15 23 28 48 50 58 84...

Official split counts:
Usage
Training       28709
PublicTest      3589
PrivateTest     3589
Name: count, dtype: int64

Emotion counts (0=Angry, 1=Disgust, 2=Fear, 3=Happy, 4=Sad, 5=Surprise, 6=Neutral):
emotion
0    4953
1     547
2    5121
3    8989
4    6077
5    4002
6    6198
Name: count, dtype: int64


## Data Preprocessing - Dataset & DataLoader

In [9]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

df['Usage'] = df['Usage'].str.strip()

def pixels_to_array(pixel_series):
    arr = np.array([np.array(p.split(), dtype=np.uint8) for p in pixel_series])
    return arr.reshape(-1, 48, 48)

class FERDataset(Dataset):
    def __init__(self, images, labels, transform=None):
        self.images = images
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = torch.from_numpy(self.images[idx]).float() / 255.0
        img = img.unsqueeze(0)
        if self.transform:
            img = self.transform(img)
        return img, int(self.labels[idx])

train_df = df[df['Usage'] == 'Training']
val_df   = df[df['Usage'] == 'PublicTest']
test_df  = df[df['Usage'] == 'PrivateTest']

X_train, y_train = pixels_to_array(train_df['pixels']), train_df['emotion'].values
X_val,   y_val   = pixels_to_array(val_df['pixels']),   val_df['emotion'].values
X_test,  y_test  = pixels_to_array(test_df['pixels']),  test_df['emotion'].values

print("Train:", X_train.shape, "| Val:", X_val.shape, "| Test:", X_test.shape)

Train: (28709, 48, 48) | Val: (3589, 48, 48) | Test: (3589, 48, 48)


In [10]:
BATCH_SIZE = 64

train_ds = FERDataset(X_train, y_train)
val_ds   = FERDataset(X_val,   y_val)
test_ds  = FERDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=2, pin_memory=True)

images, labels = next(iter(train_loader))
print("Batch images:", images.shape)
print("Batch labels:", labels.shape)
print("Pixel range:", images.min().item(), "to", images.max().item())

Batch images: torch.Size([64, 1, 48, 48])
Batch labels: torch.Size([64])
Pixel range: 0.0 to 1.0


## Sanity Checks (Forward & Backward)

In [11]:
import math
import torch.nn as nn
import torch.nn.functional as F

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("Using device:", device)

class SimpleCNN(nn.Module):
  def __init__(self, num_classes=7):
    super().__init__()
    self.conv1 = nn.Conv2d(1, 32, kernel_size=3, padding=1)
    self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
    self.pool = nn.MaxPool2d(2, 2)
    self.fc1 = nn.Linear(64 * 12 * 12, 128)
    self.fc2 = nn.Linear(128, num_classes)

  def forward(self, x):
    x = self.pool(F.relu(self.conv1(x)))
    x = self.pool(F.relu(self.conv2(x)))
    x = x.flatten(1)
    x = F.relu(self.fc1(x))
    x = self.fc2(x)
    return x

print(SimpleCNN())

Using device: cuda
SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=9216, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)


In [12]:
model = SimpleCNN().to(device)
criterion = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

with torch.no_grad():
  loss = criterion(model(images), labels)

print(f"Initial loss:      {loss.item():.4f}")
print(f"Expected (ln 7):   {math.log(7):.4f}")

Initial loss:      1.9415
Expected (ln 7):   1.9459


In [13]:
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()

images, labels = next(iter(train_loader))
images, labels = images.to(device), labels.to(device)

for step in range(200):
  optimizer.zero_grad()
  outputs = model(images)
  loss = criterion(outputs, labels)
  loss.backward()
  optimizer.step()
  if step % 40 == 0 or step == 199:
    acc = (outputs.argmax(1) == labels).float().mean().item()
    print(f"step {step:3d} | loss {loss.item():.4f} | acc {acc:.3f}")

step   0 | loss 1.9363 | acc 0.172
step  40 | loss 0.1200 | acc 1.000
step  80 | loss 0.0018 | acc 1.000
step 120 | loss 0.0007 | acc 1.000
step 160 | loss 0.0004 | acc 1.000
step 199 | loss 0.0002 | acc 1.000


## Experiment Tracking Setup (Weights & Biases)

In [14]:
!pip install -q wandb
import wandb
from google.colab import userdata

wandb.login(key=userdata.get('WANDB_API_KEY'))

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: adzid23 (adzid23-free-university-of-tbilisi-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

## Training Loop & First Experiment (SimpleCNN Baseline)

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):
  model.train()
  running_loss, correct, total = 0.0, 0, 0
  for images, labels in loader:
    images, labels = images.to(device), labels.to(device)
    optimizer.zero_grad()
    outputs = model(images)
    loss = criterion(outputs, labels)
    loss.backward()
    optimizer.step()
    running_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(1) == labels).sum().item()
    total += labels.size(0)
  return running_loss / total, correct / total

@torch.no_grad()
def evaluate(model, loader, criterion, device):
  model.eval()
  running_loss, correct, total = 0.0, 0, 0
  for images, labels in loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    loss = criterion(outputs, labels)
    running_loss += loss.item() * images.size(0)
    correct += (outputs.argmax(1) == labels).sum().item()
    total += labels.size(0)
  return running_loss / total, correct / total

In [16]:
def run_experiment(model, run_name, group, config, train_loader, val_loader, device):
  wandb.init(project="fer2013-emotion-recognition",
             name=run_name, group=group, config=config)
  model = model.to(device)
  criterion = nn.CrossEntropyLoss()
  optimizer = torch.optim.Adam(model.parameters(),
                               lr=config["lr"],
                               weight_decay=config.get("weight_decay", 0))
  for epoch in range(1, config["epochs"] + 1):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_loss, val_acc = evaluate(model, val_loader, criterion, device)
    wandb.log({"epoch": epoch,
               "train_loss": train_loss, "train_acc": train_acc,
               "val_loss": val_loss, "val_acc": val_acc})
    print(f"Epoch {epoch:2d}/{config['epochs']} | "
              f"train {train_loss:.4f}/{train_acc:.4f} | "
              f"val {val_loss:.4f}/{val_acc:.4f}")
  wandb.finish()
  return model


In [17]:
config = {
    "architecture": "SimpleCNN",
    "lr": 1e-3,
    "batch_size": 64,
    "optimizer": "Adam",
    "weight_decay": 0,
    "epochs": 25,
}

model = SimpleCNN()
run_experiment(model, run_name="SimpleCNN-baseline", group="SimpleCNN",
               config=config, train_loader=train_loader, val_loader=val_loader, device=device)

Epoch  1/25 | train 1.6256/0.3586 | val 1.4789/0.4413
Epoch  2/25 | train 1.4179/0.4553 | val 1.3691/0.4745
Epoch  3/25 | train 1.2980/0.5032 | val 1.3267/0.4921
Epoch  4/25 | train 1.2028/0.5469 | val 1.2931/0.5113
Epoch  5/25 | train 1.1182/0.5799 | val 1.3103/0.5052
Epoch  6/25 | train 1.0260/0.6205 | val 1.2917/0.5261
Epoch  7/25 | train 0.9338/0.6572 | val 1.3414/0.5135
Epoch  8/25 | train 0.8338/0.6990 | val 1.3980/0.5210
Epoch  9/25 | train 0.7276/0.7384 | val 1.4368/0.5322
Epoch 10/25 | train 0.6197/0.7818 | val 1.5927/0.5127
Epoch 11/25 | train 0.5215/0.8180 | val 1.6750/0.5227
Epoch 12/25 | train 0.4265/0.8529 | val 1.7671/0.5171
Epoch 13/25 | train 0.3421/0.8871 | val 2.0203/0.5130
Epoch 14/25 | train 0.2786/0.9089 | val 2.2183/0.5121
Epoch 15/25 | train 0.2147/0.9316 | val 2.4616/0.5208
Epoch 16/25 | train 0.1669/0.9496 | val 2.6294/0.5124
Epoch 17/25 | train 0.1418/0.9584 | val 2.8365/0.5046
Epoch 18/25 | train 0.1132/0.9682 | val 3.0793/0.5043
Epoch 19/25 | train 0.0990/0

epoch,▁▁▂▂▂▂▃▃▃▄▄▄▅▅▅▅▆▆▆▇▇▇▇██
train_acc,▁▂▃▃▃▄▄▅▅▆▆▇▇▇▇██████████
train_loss,█▇▇▆▆▅▅▄▄▄▃▃▂▂▂▁▁▁▁▁▁▁▁▁▁
val_acc,▁▄▅▆▆█▇▇█▆▇▇▇▆▇▆▆▆▆▆▆▆▆▆▇
val_loss,▁▁▁▁▁▁▁▁▁▂▂▂▃▃▄▄▅▅▆▆▆▇▇▇█
epoch,25
train_acc,0.98457
train_loss,0.05933
val_acc,0.51379
val_loss,4.12134


SimpleCNN(
  (conv1): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (conv2): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
  (pool): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  (fc1): Linear(in_features=9216, out_features=128, bias=True)
  (fc2): Linear(in_features=128, out_features=7, bias=True)
)